## CBoW Model ko train karna

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) ka hissa hai.

Is example mein, hum CBoW language model ko train karne par focus karenge taaki apna khud ka Word2Vec embedding space mil sake. Hum AG News dataset ko text ke source ke roop mein use karenge.

In [ ]:
from tensorflow import keras
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

Hum dataset load karke shuru karenge:

In [ ]:
ds_train, ds_test = tfds.load('ag_news_subset').values()

## CBoW Model

CBoW $2N$ neighboring words ke basis par ek word predict karna seekhta hai. Jaise, jab $N=1$ ho, toh sentence *I like to train networks* se hamein yeh pairs milenge: (like,I), (I, like), (to, like), (like,to), (train,to), (to, train), (networks, train), (train,networks). Yahan, pehla word neighboring word hai jo input ke roop mein use hua hai, aur doosra word woh hai jise hum predict kar rahe hain.

Agle word ko predict karne wala network banane ke liye, hamein neighboring word ko input dena hoga aur word number ko output ke roop mein lena hoga. CBoW network ka structure yeh hai:

* Input word ko embedding layer se guzaara jata hai. Yahi embedding layer hamara Word2Vec embedding hoga, isliye hum ise alag se `embedder` variable ke roop mein define karenge. Is example mein hum embedding size = 30 use karenge, halanki aap higher dimensions ke saath experiment kar sakte hain (real Word2Vec mein 300 hota hai).
* Embedding vector ko phir ek dense layer mein pass kiya jayega, jo output word predict karegi. Isliye ismein `vocab_size` neurons honge.

Keras mein embedding layer automatically jaanti hai ki numeric input ko one-hot encoding mein kaise convert karna hai, isliye hamein input word ko alag se one-hot-encode karne ki zaroorat nahi hai. Hum `input_length=1` specify karte hain taaki yeh pata chale ki hum input sequence mein sirf ek word chahte hain - aam taur par embedding layer ko lambi sequences ke saath kaam karne ke liye design kiya gaya hai.

Output ke liye, agar hum `sparse_categorical_crossentropy` ko loss function ke roop mein use karte hain, toh hamein expected result ke roop mein sirf word number dena hoga, one-hot encoding ke bina.

Hum computations ko thoda limit karne ke liye `vocab_size` ko 5000 par set karenge. Hum ek vectorizer bhi define karenge jise hum baad mein use karenge.

In [ ]:
vocab_size = 5000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(max_tokens=vocab_size,input_shape=(1,))
embedder = keras.layers.Embedding(vocab_size,30,input_length=1)

model = keras.Sequential([
    embedder,
    keras.layers.Dense(vocab_size,activation='softmax')
])

model.summary()

Chaliye vectorizer ko initialize karte hain aur vocabulary nikalte hain:

In [ ]:
def extract_text(x):
    return x['title']+' '+x['description']

vectorizer.adapt(ds_train.take(500).map(extract_text))
vocab = vectorizer.get_vocabulary()

## Training data taiyaar karna

Ab chaliye main function program karte hain, jo text se CBoW word pairs taiyaar karega. Yeh function hamein window size specify karne dega aur input aur output words ke pairs ka ek set return karega. Dhyan dein ki is function ko words par use kiya ja sakta hai, aur vectors/tensors par bhi - jisse hum text ko encode kar sakte hain, use `to_cbow` function mein pass karne se pehle.

In [ ]:
def to_cbow(sent,window_size=2):
    res = []
    for i,x in enumerate(sent):
        for j in range(max(0,i-window_size),min(i+window_size+1,len(sent))):
            if i!=j:
                res.append([sent[j],x])
    return res

print(to_cbow(['I','like','to','train','networks']))
print(to_cbow(vectorizer('I like to train networks')))

Chaliye training dataset taiyaar karte hain. Hum saari news dekhenge, `to_cbow` call karenge taaki word pairs ki list mil sake, aur un pairs ko `X` aur `Y` mein add kar denge. Time bachane ke liye, hum sirf pehle 10k news items consider karenge - agar aapke paas zyada time hai aur behtar embeddings chahiye, toh aap is limit ko aasaani se hata sakte hain :)

In [ ]:
X = []
Y = []
for i,x in zip(range(10000),ds_train.map(extract_text).as_numpy_iterator()):
    for w1, w2 in to_cbow(vectorizer(x),window_size=1):
        X.append(tf.expand_dims(w1,0))
        Y.append(tf.expand_dims(w2,0))

Hum us data ko ek dataset mein badlenge, aur training ke liye batches banayenge:

In [ ]:
ds = tf.data.Dataset.from_tensor_slices((X,Y)).batch(256)

Ab chaliye actual training karte hain. Hum `SGD` optimizer use karenge, kaafi high learning rate ke saath. Aap doosre optimizers, jaise `Adam`, ke saath bhi experiment kar sakte hain. Hum shuruaat mein 200 epochs ke liye training karenge - aur agar aap loss aur kam karna chahte hain toh is cell ko dobara run kar sakte hain.

In [ ]:
model.compile(optimizer=keras.optimizers.SGD(lr=0.1),loss='sparse_categorical_crossentropy')
model.fit(ds,epochs=200)

## Word2Vec ka use karna

Word2Vec use karne ke liye, chaliye apni vocabulary ke saare words ke vectors nikalte hain:

In [ ]:
vectors = embedder(vectorizer(vocab))
vectors = tf.reshape(vectors,(-1,30)) # we need reshape to get rid of extra dimension

In [ ]:
paris_vec = embedder(vectorizer('paris'))[0]
print(paris_vec)

Word2Vec use karke synonyms dhundhna interesting hai. Neeche wala function diye gaye input ke `n` sabse nazdeeki words return karega. Unhe dhundhne ke liye, hum $|w_i - v|$ ki value nikalte hain, jahan $v$ hamare input word ka corresponding vector hai, aur $w_i$ vocabulary mein $i$-th word ki encoding hai. Phir hum array ko sort karte hain aur `argsort` use karke corresponding indices return karte hain, aur list ke pehle `n` elements lete hain, jo vocabulary mein sabse nazdeeki words ki position ko encode karte hain.

In [ ]:
def close_words(x,n=5):
  vec = embedder(vectorizer(x))[0]
  top5 = np.linalg.norm(vectors-vec,axis=1).argsort()[:n]
  return [ vocab[x] for x in top5 ]

close_words('paris')

In [ ]:
close_words('china')

In [ ]:
close_words('official')

## Main baatein

CBoW jaisi smart techniques use karke, hum Word2Vec model train kar sakte hain. Aap skip-gram model ko bhi train karne ki koshish kar sakte hain, jo central word ke basis par neighboring word predict karne ke liye train hota hai, aur dekh sakte hain ki woh kitna achha perform karta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->